## 1-minute introduction to Jupyter ##

A Jupyter notebook consists of cells. Each cell contains either text or code.

A text cell will not have any text to the left of the cell. A code cell has `In [ ]:` to the left of the cell.

If the cell contains code, you can edit it. Press <kbd>Enter</kbd> to edit the selected cell. While editing the code, press <kbd>Enter</kbd> to create a new line, or <kbd>Shift</kbd>+<kbd>Enter</kbd> to run the code. If you are not editing the code, select a cell and press <kbd>Ctrl</kbd>+<kbd>Enter</kbd> to run the code.

---

# Lesson 14: Abstraction with Composition and Decomposition

In this lesson, we further apply the principles of abstraction to an implementation of a Battleships game to see how it can be better organised for easier understanding and to minimise the likelihood of mistakes and bugs.

## Starting code

A beginner programmer, after many hours, will usually have a working implementation that looks like this:

In [ ]:
import random

# 1. Create the grid
board = []
for i in range(10):
    row = []
    for j in range(10):
        row.append(" ")
    board.append(row)

# 2. Ship details
ships = {
    "B": 4,  # Battleship
    "C": 3,  # Cruiser
    "D": 2   # Destroyer
}
ship_locations = {}  # To track each ship’s positions

# 3. Function to place a ship
def place_ship(ship_name, size):
    placed = False

    while not placed:
        direction = random.choice(["H", "V"])
        if direction == "H":
            x = random.randint(0, 9)
            y = random.randint(0, 10 - size)

            # Check for overlap
            overlap = False
            for i in range(size):
                if board[x][y + i] != " ":
                    overlap = True
            if not overlap:
                for i in range(size):
                    board[x][y + i] = ship_name
                # Save ship location
                ship_locations[ship_name] = []
                for i in range(size):
                    ship_locations[ship_name].append((x, y + i))
                placed = True

        else:  # Vertical
            x = random.randint(0, 10 - size)
            y = random.randint(0, 9)

            overlap = False
            for i in range(size):
                if board[x + i][y] != " ":
                    overlap = True
            if not overlap:
                for i in range(size):
                    board[x + i][y] = ship_name
                ship_locations[ship_name] = []
                for i in range(size):
                    ship_locations[ship_name].append((x + i, y))
                placed = True

# 4. Place all ships
for name in ships:
    place_ship(name, ships[name])

# 5. Game board to show to player
player_board = []
for i in range(10):
    row = []
    for j in range(10):
        row.append(" ")
    player_board.append(row)

# 6. Game loop
turns = 30
hits = 0
total_ship_cells = 4 + 3 + 2

sunk_ships = []

print("Welcome to Battleships!")
while turns > 0 and hits < total_ship_cells:
    print("\nTurns left:", turns)
    print("  " + " ".join(str(i) for i in range(10)))
    for i in range(10):
        print(i, " ".join(player_board[i]))

    # 7. Player input
    guess = input("Enter your guess (row,col): ")
    if "," not in guess:
        print("Invalid format. Use row,col")
        continue

    parts = guess.split(",")
    if len(parts) != 2:
        print("Please enter two numbers.")
        continue

    try:
        row = int(parts[0])
        col = int(parts[1])
    except:
        print("Please enter valid numbers.")
        continue

    if row < 0 or row >= 10 or col < 0 or col >= 10:
        print("Coordinates out of range.")
        continue

    if player_board[row][col] != " ":
        print("You already guessed that!")
        continue

    # 8. Check hit or miss
    if board[row][col] in ships:
        ship_hit = board[row][col]
        player_board[row][col] = "X"
        hits += 1
        print("Hit!", ship_hit)

        # Remove that part from ship_locations
        ship_locations[ship_hit].remove((row, col))
        if len(ship_locations[ship_hit]) == 0 and ship_hit not in sunk_ships:
            print("You sunk the", ship_hit + "!")
            sunk_ships.append(ship_hit)

    else:
        print("Miss!")
        player_board[row][col] = "O"
        turns -= 1

# 9. End of game
print("\nGame over!")
if hits == total_ship_cells:
    print("Congratulations! You sank all the ships!")
else:
    print("You ran out of turns!")

# 10. Reveal full board
print("\nFinal board:")
print("  " + " ".join(str(i) for i in range(10)))
for i in range(10):
    row = []
    for j in range(10):
        if board[i][j] in ships and player_board[i][j] == " ":
            row.append(board[i][j])
        else:
            row.append(player_board[i][j])
    print(i, " ".join(row))


The game is fine; it works after all ... for now? But if you come across a bug, or want to add a feature, and come back to this code after 6 months, it'll be a slog to try to understand what you were thinking when you wrote it. And that just makes you less motivated to fix that bug or add that feature ... is there a way to make it easier?

## Make smaller, composable functions

The game code only has one function, and it's a huge one. If you come across a bug, it's hard to know where to begin bug-hunting. Instead, if we have smaller functions, it's easier to work through the logic of a single function.

Let's break the code up into the following functions, each one representing a single task of the game:

- `create_grid()`
- `place_ship()`
- `run_game()`
- `display_board()`
- `prompt_player_input()`
- `is_valid_input()`
- `is_valid_guess()`
- `prompt_valid_guess()`
- `get_enemy_guess()`
- `is_target_hit()`
- `targetting_update()`
- `player_update()`
- `is_gameover()`
- `display_final_boards()`

Even before we put in any real code, we can see the naming of the functions is important. Naming things well helps us to understand what they do even before we dig into the documentation.

The functions are even more explanatory once we put in the parameter names, types, and docstrings:

In [ ]:
def create_grid(n: int, char: str) -> list[list[str]]:
    """Create a grid of size n x n filled with char, and return it."""
    pass

def place_ship(board: list[list[str]], ship_name: str, size: int) -> None:
    """Place a ship of size size on the board at a random location."""
    pass

def run_game(turns: int, ships: dict[str, int]) -> None:
    """Run the main game loop."""
    pass

def display_board(board: list[list[str]]) -> None:
    """Display the current state of the board."""
    pass

def is_valid_guess(board: list[list[str]], text: str) -> bool:
    """Check if the guess is valid. Return True if valid, False otherwise."""
    pass

def prompt_valid_guess(board: list[list[str]]) -> tuple[int, int]:
    """Prompt the user for a valid guess.

    If the guess is invalid, keep prompting until a valid guess is entered.

    Return the row and column of the guess, as a tuple.
    """
    pass

def get_enemy_guess(board: list[list[str]]) -> tuple[int, int]:
    """Generate a random guess for the enemy.

    Return the row and column of the guess, as a tuple.
    """
    pass

def is_target_hit(board: list[list[str]], x: int, y: int) -> bool:
    """Check if the target hit a ship. Return True if it did, False otherwise."""
    pass

def targetting_update(board: list[list[str]], hit_what: str, x: int, y: int) -> None:
    """Update the targetting board with the guess."""
    pass

def player_update(board: list[list[str]], x: int, y: int) -> None:
    """Update the player board with the guess."""
    pass

def is_gameover(turns: int, player_hits: int, enemy_hits: int, total_ship_cells: int) -> bool:
    """Check if the game is over. Return True if it is, False otherwise."""
    pass

def display_final_boards(targetting: list[list[str]], board: list[list[str]]) -> None:
    """Display the final boards: targetting overlaid on the ship board."""
    pass

## Composing functions: Part 1

Before we start to implement the functions, have a look at the *interface* of the functions—their parameter types and output types—and see if you are able to use them even before you see the code that implements them. For now, let's pretend they work perfectly.

Take some time to actually do this before you scroll down to see the sample code.

Really.

Try it first.

.  
.  
.  
.  
.  
.  
.  
.  
.  
.  
.  
.  
.  
.  

Tried it?

Okay, scroll down and see if this is what you had in mind:

In [ ]:
import random

# 2. Ship details
ships = {
    "B": 4,  # Battleship
    "C": 3,  # Cruiser
    "D": 2   # Destroyer
}

run_game(turns=30, ships={"B": 4, "C": 3, "D": 2})

Okay, that wasn't very clear, all the code ended up hidden in `run_game()` huh. Let's have a look inside `run_game()` then, since that's where the action happens:

In [ ]:
# inside run_game(); pretend all the code below is indented by 4 spaces
# Variables for tracking game state
player_hits = 0
enemy_hits = 0
total_ship_cells = sum(ships.values())

# Data structures for tracking game data
# For tracking player ships and damage
player_board = create_grid(10, " ")
player_ship_cells = {}  # To track each ship’s remaining cells
player_targetting = create_grid(10, " ")  # For tracking player guesses and hits
player_sunk_ships = []

# Similarly for the enemy
enemy_board = create_grid(10, " ")
enemy_ship_cells = {}
enemy_targetting = create_grid(10, " ")
enemy_sunk_ships = []

# Place ships on the boards
for name, size in ships.items():
    place_ship(player_board, name, size)
    player_ship_cells[name] = size
    place_ship(enemy_board, name, size)
    enemy_ship_cells[name] = size

while not is_gameover(turns, player_hits, enemy_hits, total_ship_cells):
    print("\nTurns left:", turns)
    print("\nPlayer's turn")
    display_board(player_targetting)
    x, y = prompt_valid_guess(player_targetting)

    hit_char = enemy_board[x][y]
    targetting_update(player_targetting, hit_char, x, y)
    if is_target_hit(enemy_board, x, y):
        player_hits += 1
        enemy_ship_cells[hit_char].remove((x, y))
        if len(enemy_ship_cells[hit_char]) == 0:
            print("You sunk the", hit_char + "!")
            player_sunk_ships.append(hit_char)

    print("\nEnemy's turn")
    display_board(player_board)
    x, y = get_enemy_guess(player_board)

    hit_char = player_board[x][y]
    targetting_update(enemy_targetting, hit_char, x, y)
    if is_target_hit(player_board, x, y):
        enemy_hits += 1
        player_ship_cells[hit_char].remove((x, y))
        if len(player_ship_cells[hit_char]) == 0:
            print("Enemy sunk the", hit_char + "!")
            enemy_sunk_ships.append(hit_char)
    turns -= 1

# Game is over
if player_hits == total_ship_cells:
    display_final_boards(player_targetting, enemy_board)
    print("Congratulations! You sank all the enemy ships!")
elif enemy_hits == total_ship_cells:
    display_final_boards(enemy_targetting, player_board)
    print("Game over! The enemy sank all your ships!")
else:
    print("Game over! You ran out of turns!")

### Observations

- We haven't even implemented any of the functions yet, but it's already easier to see at a glance the logic of the game and how it's going to work. Certainly more easily than with the first unabstracted version of the code.

- In this abstracted code, there are no indexes. We avoid dealing with indexes when looking at "high-level" code, i.e. the general logic of the game, because indexes are a "low-level" detail, i.e. it delves into the details of how parts of the game work.

- In high-level code, there are also very few `print()` function calls. Most of them are hidden in `display_*()` functions so as not to clutter up the code and obscure its high-level logic.

- Breaking up a huge task into smaller tasks, each one handled by a single function, makes each individual function easier to reason about and implement.

- Using functions to chunk logical blocks of code enables us to invoke repetitive operations without adding much more code. For example, we run many of the functions for both the player and the enemy, without having to duplicate the code. This reduces the chance of typos, and standardises operations in a way that makes debugging easier.

- Despite the above benefits, we still notice some repetitive code, particularly within the game loop.

## Decomposing functions

Can we break down some of these tasks into even smaller functions? E.g. I'm noticing that `is_gameover()` is particularly huge, involving 4 different arguments.

Thinking through its sub-operations, it needs to:
- check if player has won
- check if enemy has won
- check if we are out of turns

We could thus further decompose the gameover check into:

- `is_player_won()`
- `is_enemy_won()`

The implementation of `is_gameover()` might look like:

```python
def is_gameover(turns, player_hits, enemy_hits, total_ship_cells) -> bool:
    return (
        is_player_won(player_hits, total_ship_cells)
        or is_enemy_won(enemy_hits, total_ship_cells)
        or turns == 0
    )
```

As a side benefit, we can reuse these two functions further after the loop:

```python
# Game is over
if is_player_won(player_hits, total_ship_cells):
    display_final_boards(player_targetting, enemy_board)
    print("Congratulations! You sank all the enemy ships!")
elif is_enemy_won(enemy_hits, total_ship_cells):
    display_final_boards(enemy_targetting, player_board)
    print("Game over! The enemy sank all your ships!")
else:
    print("Game over! You ran out of turns!")
```

That doesn't seem like much of a benefit ... the line looks even longer now! But what we've done is reduce the amount of low-level code; if we later add more things to the game and need to modify the victory logic, we only need to change it in one place, the `is_player_won()` and `is_enemy_won()` functions (can we further abstract these two functions to one function?).

### Another example

What about `place_ship()`? The implementation for this was particularly long in the original code, due to having to handle horizontal and vertical cases separately.

It could certainly be easier to read and debug if we split it up into `place_ship_horizontal()` and `place_ship_vertical()`. Sometimes there's a bug that only affects vertical but not horizontal placement; having two smaller functions makes it easier for us to narrow down the problem zone and eliminate contributing factors.

## Labelling constants by name (instead of value)

If you return to the original code 6 months later, would you remember what `10` and `" "` in the original were for? Space strings (`" "`) are especially common in code, and it's easy to mix up what we are using them for (e.g. to space out the grid, but also to represent an empty grid)

Let's label them instead. Many programming conventions ask programmers to state the constants upfront, near the top of the file, so let's do that:

In [ ]:
GRID_SIZE = 10
EMPTY = " "
HIT = "X"
MISS = "O"

player_board = create_grid(10, EMPTY)

It's easier to see what the second argument to `create_grid()` is for, isn't it? Now it's easy to see that the char is (most likely) used to fill the grid. And we also won't be left guessing if `'X'` represents a hit or miss.

## Remaining problems

Despite the above improvements, some problems remain:

- It is tricky to keep track of the data for the player and the enemy. There's a targetting grid, a board grid, a list of sunk ships, a dict of ship cells, the number of hits, ... and possibly other variables that were introduced which slipped our notice.
- Different functions call for different data: `is_gameover()` requires the number of hits, `targetting_update()` requires the targetting grid, `is_valid_guess()` requires the board grid, and so on.  
If only there was a way to **bundle up** player and enemy data separately, and simply pass that around to each function ...
- It is easy to make a typo and end up passing the wrong argument to a function, which could mess things up really badly and make things difficult to debug. What happens if we pass a board grid to `targetting_update()` accidentally? Or pass the player's targetting grid instead of the enemy's? Because the datatype is still the same—a list of lists—we wouldn't get an error, and instead get a confusing result, wasting lots of time debugging.  
If only there was a way to **ensure correct data** is always passed ...
- Did you spot the bug in the original code? We access a cell in the grid using `grid[x][y]` notation, but because we display each inner list as a row, we should actually use `grid[y][x]`. Fixing this bug isn't the worst thing: it's making sure that the fix is applied everywhere in the code consistently. Missing out even one instance means we have a difficult-to-identify bug, especially if we thought we'd fixed it previously. Who wants to waste time chasing bugs?  
Worse, this usage isn't intuitive; we are so used to referring to coordinates by `(x, y)` notation that we are sure to accidentally type `grid[x][y]` again at some point.  
If only there was a way we could write (x, y) but have the indexing be done (y, x) instead ... could we **separate the interface from how it is implemented**?

We'll explore one kind of programming that aims to address these issues in the next chapter.